In [1]:
import pandas as pd
#Load the raw CSV dataset with ISO encoding to handle special characters
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='ISO-8859-1')
print(f"Original shape: {df.shape}")

Original shape: (180519, 53)


In [2]:
#Converting headers to clean snake_case and strip special characters for PostgreSQL compatability
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('/', '_', regex=False)
)
print(df.columns[:10]) #Check cleaned column names

Index(['type', 'days_for_shipping_real', 'days_for_shipment_scheduled',
       'benefit_per_order', 'sales_per_customer', 'delivery_status',
       'late_delivery_risk', 'category_id', 'category_name', 'customer_city'],
      dtype='str')


In [3]:
# Drop non-essential and PII columns (Personally Identifiable Information) columns to enhance security
drop_cols = [
    'customer_email',
    'customer_password',
    'product_image',
    'customer_street',
    'customer_zipcode',
    'order_zipcode'
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"Shape after dropping PII columns: {df.shape}")

Shape after dropping PII columns: (180519, 47)


In [4]:
# Filling missing last names with placeholder
df['customer_lname'] = df['customer_lname'].fillna('Unknown')

# Converting text dates into proper ISO Datetime objects
df['order_date'] = pd.to_datetime(df['order_date_dateorders'])
df['shipping_date'] = pd.to_datetime(df['shipping_date_dateorders'])

print("Date conversion complete!")

Date conversion complete!


In [5]:
!pip install psycopg2-binary

In [6]:
import os
from sqlalchemy import create_engine

# 1. Save local backup CSV
df.to_csv('cleaned_supply_chain.csv', index=False)

# 2. Database credentials (uses environment variable or default local fallback)
db_user = os.getenv('DB_USER', 'postgres')
db_password = os.getenv('DB_PASS', 'Password') 
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'supply_chain_db')

# 3. Create SQLAlchemy connection engine
engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

# 4. Ingest clean DataFrame into PostgreSQL table
df.to_sql('fact_supply_chain', engine, if_exists='replace', index=False)

print("Phase 1 Complete! Data successfully loaded into PostgreSQL.")

Phase 1 Complete! Data successfully loaded into PostgreSQL.
